# ARK Invest: Abilità o Fortuna?

In [2]:
# ── Import standard ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import fmfinance as fm
import quantstats as qs
import time
import random
import warnings

# statsmodels
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.compat import lzip
from statsmodels.regression.rolling import RollingOLS
from statsmodels.iolib.summary2 import summary_col
from statsmodels.datasets import longley

# scipy
from scipy.stats import norm
from scipy.stats import linregress
from scipy import stats

# datetime e pandas utils
from datetime import datetime
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# PyPortfolioOpt (Es02)
from pypfopt import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns

# Widget e display (Es02b, Es04b)
import ipywidgets as widgets
from IPython.display import display

# Web scraping (Es03c)
import requests
from bs4 import BeautifulSoup
from io import StringIO

# Soppressione warning
warnings.filterwarnings("ignore", category=FutureWarning)

import plotly.express as px

## PREPARAZIONE DEI DATI

In [18]:
start = '2019-01-01'
end = '2024-12-31'

ff4f = fm.ff('4_factors', start, end)[0]

tickers = ['ARKK', 'ARKW', 'ARKG', 'ARKF', 'ARKQ', 'SPY']
prices = yf.download(tickers, start=start, end=end,
                     auto_adjust=False, multi_level_index=False, interval='1mo')['Adj Close']

display(ff4f)
display(prices)

[*********************100%***********************]  6 of 6 completed


,Mkt-RF,SMB,HML,Mom,RF
Date,,,,,
2019-01-01,8.37,2.76,-0.39,-8.62,0.21
2019-02-01,3.42,2.05,-2.66,0.50,0.18
2019-03-01,1.10,-3.03,-4.14,2.24,0.19
2019-04-01,3.97,-1.74,2.13,-2.91,0.21
2019-05-01,-6.92,-1.26,-2.45,7.55,0.21
...,...,...,...,...,...
2024-08-01,1.60,-3.50,-1.10,4.86,0.48
2024-09-01,1.72,-0.12,-2.77,-0.56,0.40
2024-10-01,-1.00,-0.96,0.89,2.94,0.39


Ticker,ARKF,ARKG,ARKK,ARKQ,ARKW,SPY
Date,,,,,,
2019-01-01,NaN,27.011171,41.523796,32.629158,46.841206,242.095993
2019-02-01,20.852493,29.461514,44.909355,34.610847,49.130287,249.943771
2019-03-01,21.196594,31.471558,45.073326,33.394371,49.007328,253.351959
2019-04-01,22.278053,31.308836,45.545963,33.959446,50.482933,264.863434
2019-05-01,20.685360,27.872620,39.286041,28.783514,45.015606,247.972763
...,...,...,...,...,...,...
2024-08-01,28.275276,26.389999,44.820000,56.085991,77.984314,552.062927
2024-09-01,29.793949,25.600000,47.529999,60.763977,83.833870,561.935120
2024-10-01,30.843033,23.139999,45.889999,60.823826,85.636009,558.628967


In [24]:
returns    = prices.pct_change()*100
returns.dropna(inplace=True)
returns

Ticker,ARKF,ARKG,ARKK,ARKQ,ARKW,SPY
Date,,,,,,
2019-03-01,1.650167,6.822610,0.365115,-3.514726,-0.250272,1.363582
2019-04-01,5.102042,-0.517043,1.048596,1.692126,3.010988,4.543669
2019-05-01,-7.149158,-10.975229,-13.744186,-15.241509,-10.830050,-6.377124
2019-06-01,7.652092,18.372261,17.800157,15.064750,9.287661,6.440981
2019-07-01,0.958059,0.319111,1.000407,-2.073456,0.903675,2.005668
...,...,...,...,...,...,...
2024-08-01,3.967675,-2.077927,-1.559409,-0.530686,1.799718,2.336543
2024-09-01,5.371028,-2.993554,6.046406,8.340739,7.500939,1.788237
2024-10-01,3.521130,-9.609379,-3.450451,0.098494,2.149655,-0.588351


In [26]:
rf = ff4f['RF']
er = returns.subtract(rf, axis=0).dropna()

display(er.round(2))

Ticker,ARKF,ARKG,ARKK,ARKQ,ARKW,SPY
Date,,,,,,
2019-03-01,1.46,6.63,0.18,-3.70,-0.44,1.17
2019-04-01,4.89,-0.73,0.84,1.48,2.80,4.33
2019-05-01,-7.36,-11.19,-13.95,-15.45,-11.04,-6.59
2019-06-01,7.47,18.19,17.62,14.88,9.11,6.26
2019-07-01,0.77,0.13,0.81,-2.26,0.71,1.82
...,...,...,...,...,...,...
2024-08-01,3.49,-2.56,-2.04,-1.01,1.32,1.86
2024-09-01,4.97,-3.39,5.65,7.94,7.10,1.39
2024-10-01,3.13,-10.00,-3.84,-0.29,1.76,-0.98


## FASE 1: Return Analysis